# LangGraph Planner 비교 실험

## Experiment Goal

- Random, BFS, LLM, LLM+BFS Search Guard를 동일한 레벨과 seed에서 비교한다.
- 성공률, 행동 수, 계획 오류·재시도, 알고리즘 폴백과 LLM 응답 시간을 측정한다.
- 저장 출력은 없다. 위에서 아래로 실행해 현재 결과를 만든다.

## Context & Methods

상태 배열을 표준 Sokoban 문자 보드로 직렬화하고, 모델에는 `{"action":"UP"}` 형태의 JSON Schema 출력을 요구한다. LangGraph가 응답 형식, 이동 가능 여부와 재시도 흐름을 검증·실행한다.

### Key Assumptions

- `.env`의 Ollama 서버와 모델이 실행 시점에 사용 가능하다.
- `temperature=0`과 episode seed를 모델 호출에 전달하지만, 런타임·모델 버전에 따라 완전한 결정성은 보장되지 않는다.
- `heldout-turn`은 제품 기본 레벨에 포함하지 않은 실험용 소형 레벨이다.
- 실행 시간은 로컬/원격 Ollama 상태의 영향을 받는다.

### 1. Load dependencies and fixed parameters

In [ ]:
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from matplotlib import animation

from sokoban_agent.env import (
    DEFAULT_LEVELS,
    FixedLevelProvider,
    SokobanEnv,
    parse_level,
)
from sokoban_agent.evaluation import (
    run_benchmark_traces,
    summarize_by_planner,
)
from sokoban_agent.planning import (
    BFSPlanner,
    LLMPlanner,
    RandomPlanner,
    SearchGuardPlanner,
)
from sokoban_agent.planning.llm import (
    OllamaClient,
    OllamaSettings,
)

LEVEL_IDS = ['tiny-push', 'tiny-walk', 'heldout-turn']
SEEDS = [0, 1]
MAX_STEPS = 15
MAX_ATTEMPTS = 3

settings = OllamaSettings.from_env()
experiment_config = {
    'model': settings.model,
    'temperature': settings.temperature,
    'level_ids': LEVEL_IDS,
    'seeds': SEEDS,
    'max_steps': MAX_STEPS,
    'max_planning_attempts': MAX_ATTEMPTS,
}
experiment_config

## Data

### 2. Build the fixed and held-out level cohort

In [ ]:
heldout_level = parse_level(
    'heldout-turn',
    [
        '#####',
        '#.  #',
        '# $ #',
        '#  @#',
        '#####',
    ],
)
level_provider = FixedLevelProvider(
    [
        DEFAULT_LEVELS.get('tiny-push'),
        DEFAULT_LEVELS.get('tiny-walk'),
        heldout_level,
    ]
)
LEVEL_IDS

### 3. Run identical cases

In [ ]:
client = OllamaClient(settings)
planners = [
    RandomPlanner(),
    BFSPlanner(),
    LLMPlanner(
        client,
        model_name=settings.model,
    ),
    SearchGuardPlanner(
        LLMPlanner(client, model_name=settings.model)
    ),
]
env = SokobanEnv(
    level_provider=level_provider,
    max_steps=MAX_STEPS,
)
try:
    traces = run_benchmark_traces(
        env,
        planners,
        level_ids=LEVEL_IDS,
        seeds=SEEDS,
        max_planning_attempts=MAX_ATTEMPTS,
    )
finally:
    env.close()

results = [trace.result for trace in traces]
results_df = pd.DataFrame.from_records(
    asdict(result) for result in results
)
results_df

## Results

### 4. Compare success, efficiency, recovery, and latency

In [ ]:
summary_df = pd.DataFrame.from_records(
    asdict(summary) for summary in summarize_by_planner(results)
).set_index('planner_name')
summary_columns = [
    'episode_count',
    'success_count',
    'success_rate',
    'mean_actions',
    'mean_actions_on_success',
    'mean_elapsed_seconds',
    'total_algorithm_calls',
    'total_algorithm_fallbacks',
    'total_llm_calls',
    'total_llm_retries',
    'total_llm_format_errors',
    'total_llm_invalid_actions',
    'mean_llm_elapsed_seconds',
]
summary_df[summary_columns]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
summary_df['success_rate'].plot.bar(
    ax=axes[0], ylim=(0, 1), title='Success rate'
)
summary_df['mean_actions'].plot.bar(
    ax=axes[1], title='Mean actions per episode'
)
summary_df['mean_elapsed_seconds'].plot.bar(
    ax=axes[2], title='Mean elapsed seconds'
)
for axis in axes:
    axis.set_xlabel('Graph planner')
    axis.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()

### 5. Inspect LLM failures by level and seed

In [ ]:
llm_name = f'graph:llm:{settings.model}'
hybrid_name = f'graph:hybrid:{llm_name}+bfs'
llm_columns = [
    'level_id',
    'seed',
    'success',
    'deadlock',
    'truncated',
    'action_count',
    'llm_calls',
    'llm_retries',
    'llm_client_errors',
    'llm_format_errors',
    'llm_invalid_actions',
    'llm_elapsed_seconds',
    'failure_reason',
]
llm_results_df = results_df[
    results_df['planner_name'] == llm_name
][llm_columns]
llm_results_df

### 6. Replay exact agent movement

기본 선택은 장기 계획 실패가 드러나는 LLM의 `tiny-walk`, seed 0 에피소드다. `ANIMATION_CASE`를 바꾸면 같은 실험의 다른 Planner 경로도 재생할 수 있다. 표와 영상에는 환경이 실제로 실행한 행동만 포함된다.

In [ ]:
ANIMATION_CASE = {
    'planner_name': llm_name,
    'level_id': 'tiny-walk',
    'seed': 0,
}
selected_trace = next(
    trace
    for trace in traces
    if trace.result.planner_name == ANIMATION_CASE['planner_name']
    and trace.result.level_id == ANIMATION_CASE['level_id']
    and trace.result.seed == ANIMATION_CASE['seed']
)

action_log_df = pd.DataFrame.from_records(
    {
        'frame': frame.index,
        'action': (
            frame.action.name
            if frame.action is not None
            else 'RESET'
        ),
        'invalid_move': frame.invalid_move,
        'pushed': frame.pushed,
        'success': frame.success,
        'deadlock': frame.deadlock,
    }
    for frame in selected_trace.frames
)
action_log_df

In [ ]:
RGB_PALETTE = np.asarray(
    [
        [238, 238, 238],  # floor
        [45, 45, 45],     # wall
        [255, 196, 64],   # target
        [139, 90, 43],    # box
        [55, 126, 184],   # player
        [76, 175, 80],    # box on target
        [126, 87, 194],   # player on target
    ],
    dtype=np.uint8,
)

def animate_episode_trace(trace, interval_ms=700):
    fig, axis = plt.subplots(figsize=(4.5, 4.5))
    image = axis.imshow(
        RGB_PALETTE[trace.frames[0].observation]
    )
    axis.set_xticks([])
    axis.set_yticks([])
    title = axis.set_title('')

    def update(frame_index):
        frame = trace.frames[frame_index]
        image.set_data(RGB_PALETTE[frame.observation])
        action_name = (
            frame.action.name
            if frame.action is not None
            else 'RESET'
        )
        if frame.success:
            status = 'SUCCESS'
        elif frame.deadlock:
            status = 'DEADLOCK'
        elif frame_index == len(trace.frames) - 1:
            status = (
                'STEP LIMIT'
                if trace.result.truncated
                else 'STOPPED'
            )
        else:
            status = 'RUNNING'
        title.set_text(
            f"{trace.result.planner_name} | "
            f"{trace.result.level_id} | "
            f"frame {frame.index} | {action_name} | {status}"
        )
        return image, title

    episode_animation = animation.FuncAnimation(
        fig,
        update,
        frames=len(trace.frames),
        interval=interval_ms,
        repeat=True,
        blit=False,
    )
    html = episode_animation.to_jshtml(default_mode='loop')
    plt.close(fig)
    return HTML(html)

display(animate_episode_trace(selected_trace))

### 7. Validate the comparison cohort

In [ ]:
case_sets = {
    planner_name: set(
        zip(group['level_id'], group['seed'], strict=True)
    )
    for planner_name, group in results_df.groupby('planner_name')
}
expected_cases = set(
    (level_id, seed)
    for level_id in LEVEL_IDS
    for seed in SEEDS
)
assert all(cases == expected_cases for cases in case_sets.values())
assert summary_df.loc['graph:bfs', 'success_rate'] == 1.0
assert summary_df.loc[llm_name, 'total_llm_client_errors'] == 0
assert summary_df.loc[hybrid_name, 'total_algorithm_calls'] > 0
assert len(selected_trace.frames) == (
    selected_trace.result.action_count + 1
)
assert action_log_df['action'].eq('RESET').sum() == 1
{name: len(cases) for name, cases in case_sets.items()}

## Takeaways

In [ ]:
llm_summary = summary_df.loc[llm_name]
hybrid_summary = summary_df.loc[hybrid_name]
bfs_summary = summary_df.loc['graph:bfs']
solved_levels = llm_results_df.loc[
    llm_results_df['success'], 'level_id'
].value_counts().to_dict()
display(Markdown(
    f'- LLM은 {int(llm_summary.success_count)}/'
    f'{int(llm_summary.episode_count)} 에피소드를 해결했다 '
    f'({llm_summary.success_rate:.0%}).\n'
    f'- BFS는 같은 cohort에서 '
    f'{bfs_summary.success_rate:.0%} 성공했다.\n'
    f'- LLM+BFS Search Guard는 '
    f'{hybrid_summary.success_rate:.0%} 성공했고, BFS 폴백은 '
    f'{int(hybrid_summary.total_algorithm_fallbacks)}회였다.\n'
    f'- LLM의 총 호출은 {int(llm_summary.total_llm_calls)}회, '
    f'재시도는 {int(llm_summary.total_llm_retries)}회, '
    f'막힌 행동 거절은 '
    f'{int(llm_summary.total_llm_invalid_actions)}회였다.\n'
    f'- 성공 레벨별 횟수: `{solved_levels}`. 이 결과는 소형 '
    '3개 레벨의 탐색 실험이며 전체 Boxoban 성능을 대표하지 않는다.'
))